In [1]:
import torch

print(f"Is CUDA actually awake?: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"We are running on: {device}")

Is CUDA actually awake?: True
We are running on: cuda


In [2]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, TrainingArguments, Trainer

print("May the GPU gods have mercy on your soul...")

# 1. Load the RAW Tabular Data because we actually need the text to train the text model
df_train = pd.read_csv("../data/train.csv")
df_valid = pd.read_csv("../data/valid.csv")

# 🚨 SURVIVAL TIP 🚨
# Do NOT train on all 240k rows. It will take 14 hours and you will miss the deadline.
# Take a random sample of 50k rows. That's more than enough for BERT to learn the vibes.
df_train = df_train.sample(90000, random_state=42)

# Create the text inputs and use the LOG-TRANSFORMED salaries as the labels
train_texts = (df_train['Title'].fillna("") + " | " + df_train['FullDescription'].fillna("")).tolist()
train_labels = np.log1p(df_train['SalaryNormalized'].values).astype(np.float32)

valid_texts = (df_valid['Title'].fillna("") + " | " + df_valid['FullDescription'].fillna("")).tolist()
valid_labels = np.log1p(df_valid['SalaryNormalized'].values).astype(np.float32)

# 2. Setup Tokenizer and Model
# num_labels=1 tells BERT we are doing Regression (predicting a single continuous number)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
# Load the model and immediately lock it to the GPU
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=1).to(device)
# 3. Convert to Hugging Face Dataset format
def create_hf_dataset(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
    return Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'label': labels # Trainer automatically looks for a column named 'label'
    })

print("Tokenizing the text...")
train_dataset = create_hf_dataset(train_texts, train_labels)
valid_dataset = create_hf_dataset(valid_texts, valid_labels)


c:\Users\nishk\anaconda3\envs\torch_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


May the GPU gods have mercy on your soul...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5988.70it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizing the text...


In [3]:

# 4. The God-Tier Training Arguments for 6GB VRAM
training_args = TrainingArguments(
    output_dir='../data/results',
    eval_strategy="epoch",
    learning_rate=2e-5,               # Keep it low so it doesn't forget English
    per_device_train_batch_size=8,    # TINY batch size so your VRAM doesn't explode
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,    # Fakes a batch size of 32 (8 * 4)
    num_train_epochs=2,               # 2 is enough. Any more and it overfits.
    weight_decay=0.01,
    fp16=True,                        # MIXED PRECISION: Saves 50% VRAM. Crucial for RTX 4050.
    save_strategy="epoch",
    logging_steps=100
)

# 5. Build a custom metric to track MAE during training
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Reverse the log so the MAE makes sense in real money
    preds_real = np.expm1(predictions).flatten()
    labels_real = np.expm1(labels)
    mae = np.mean(np.abs(preds_real - labels_real))
    return {"mae": mae}

# 6. Train the Beast
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)

print("Starting training. Touch grass while you wait.")
trainer.train()

# 7. Save the Fine-Tuned Model
trainer.save_model("../data/my_finetuned_distilbert")
tokenizer.save_pretrained("../data/my_finetuned_distilbert")
print("Model saved! You survived.")

Starting training. Touch grass while you wait.


Epoch,Training Loss,Validation Loss,Mae
1,0.636139,0.085099,9138.666016
2,0.583133,0.063181,7060.777832


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.17it/s]

Model saved! You survived.
